ランダムネス小

In [3]:
import numpy as np
import random
import pandas as pd
from scipy.sparse import csr_matrix, hstack, vstack
import time

class DihedralGirthOptimizer:
    def __init__(self, n=384):
        self.n = n
        self.P = 2 * n  # 768
        self.central = n // 2

    def get_element(self, k_a, k_b, is_ref_a=False, is_ref_b=False):
        """領域AとBで個別に作用を定義する二面体群の元を生成"""
        p = np.arange(self.P)
        # 領域 A (0 ~ n-1)
        if not is_ref_a:
            p[0:self.n] = (p[0:self.n] + k_a) % self.n
        else:
            p[0:self.n] = (self.n - p[0:self.n] + k_a) % self.n
        # 領域 B (n ~ 2n-1)
        if not is_ref_b:
            p[self.n:self.P] = self.n + (p[self.n:self.P] - self.n + k_b) % self.n
        else:
            p[self.n:self.P] = self.n + (self.n - (p[self.n:self.P] - self.n) + k_b) % self.n
        return p

    def check_commute(self, p1, p2):
        """2つの置換が可換かチェック"""
        # x -> p1(p2(x)) == x -> p2(p1(x))
        return np.array_equal(p1[p2], p2[p1])

    def has_c4(self, p1, p2, p3, p4):
        """2x2ブロックに長さ4のサイクルがあるか(不動点チェック)"""
        # Π = p4^-1 * p3 * p2^-1 * p1
        # 実装上は p3(p2_inv(p1(x))) == p4(x) をチェック
        inv_p2 = np.zeros_like(p2)
        inv_p2[p2] = np.arange(self.P)
        
        res_left = p3[inv_p2[p1]]
        return np.any(res_left == p4)

    def solve_step(self):
        """混合可換条件を満たすF, Gのセットを1つ生成"""
        F = []
        G = []
        
        # Fブロックの生成
        for i in range(6):
            ka, kb = random.randint(1, self.n-1), random.randint(1, self.n-1)
            if i == 0:   # f0: Aで反転, Bで回転
                F.append(self.get_element(ka, kb, is_ref_a=True, is_ref_b=False))
            elif i == 1: # f1: Aで回転, Bで反転
                F.append(self.get_element(ka, kb, is_ref_a=False, is_ref_b=True))
            else:        # その他: 両領域で回転
                F.append(self.get_element(ka, kb, is_ref_a=False, is_ref_b=False))

        # Gブロックの生成
        for j in range(6):
            # 可換性を守るための固定指数
            if j == 2:   # g2: Aで中心元(f0と可換), Bで一般回転(f1と非可換)
                ka, kb = self.central, random.randint(1, self.n-1)
                while kb == self.central: kb = random.randint(1, self.n-1)
            elif j == 3: # g3: Aで一般回転(f0と非可換), Bで中心元(f1と可換)
                ka, kb = random.randint(1, self.n-1), self.central
                while ka == self.central: ka = random.randint(1, self.n-1)
            else:        # その他: 両領域で中心元 (全Fと可換)
                ka, kb = self.central, self.central
            G.append(self.get_element(ka, kb, is_ref_a=False, is_ref_b=False))
        
        return F, G

    def optimize(self, max_trials=100000, J=3):
        print(f"探索開始: J={J}, P={self.P}")
        start_time = time.time()
        
        for trial in range(1, max_trials + 1):
            F, G = self.solve_step()
            
            # プロトグラフに基づいた 3x12 行列の構築 (Hx)
            L_h = 6
            blocks = []
            for i in range(J):
                row = []
                for j in range(L_h): row.append(F[(j - i) % L_h])
                for j in range(L_h): row.append(G[(j - i) % L_h])
                blocks.append(row)
            
            # C4チェック
            c4_count = 0
            M, N = J, 2 * L_h
            for i1 in range(M):
                for i2 in range(i1 + 1, M):
                    for j1 in range(N):
                        for j2 in range(j1 + 1, N):
                            if self.has_c4(blocks[i1][j1], blocks[i2][j1], blocks[i2][j2], blocks[i1][j2]):
                                c4_count += 1
                                break # 1つでもあればこのtrialは失敗
                        if c4_count > 0: break
                if c4_count > 0: break
            
            if c4_count == 0:
                print(f"成功! 試行回数: {trial}, 時間: {time.time()-start_time:.2f}s")
                return F, G
            
            if trial % 100 == 0:
                print(f"Trial {trial}...")
        
        print("解が見つかりませんでした。")
        return None, None

# 実行
optimizer = DihedralGirthOptimizer(n=384)
F_res, G_res = optimizer.optimize()

if F_res:
    # 可換表の最終確認
    matrix = np.zeros((6, 6), dtype=int)
    for i in range(6):
        for j in range(6):
            matrix[i, j] = 1 if optimizer.check_commute(F_res[i], G_res[j]) else 0
    
    df = pd.DataFrame(matrix, index=[f'f{i}' for i in range(6)], columns=[f'g{j}' for j in range(6)])
    print("\n--- 探索結果：混合可換表 ---")
    print(df)
    print("\nC4 count is confirmed to be 0 for this configuration.")

探索開始: J=3, P=768
Trial 100...
Trial 200...
Trial 300...
Trial 400...
Trial 500...
Trial 600...
Trial 700...
Trial 800...
Trial 900...
Trial 1000...
Trial 1100...
Trial 1200...
Trial 1300...
Trial 1400...
Trial 1500...
Trial 1600...
Trial 1700...
Trial 1800...
Trial 1900...
Trial 2000...
Trial 2100...
Trial 2200...
Trial 2300...
Trial 2400...
Trial 2500...
Trial 2600...
Trial 2700...
Trial 2800...
Trial 2900...
Trial 3000...
Trial 3100...
Trial 3200...
Trial 3300...
Trial 3400...
Trial 3500...
Trial 3600...
Trial 3700...
Trial 3800...
Trial 3900...
Trial 4000...
Trial 4100...
Trial 4200...
Trial 4300...
Trial 4400...
Trial 4500...
Trial 4600...
Trial 4700...
Trial 4800...
Trial 4900...
Trial 5000...
Trial 5100...
Trial 5200...
Trial 5300...
Trial 5400...
Trial 5500...
Trial 5600...
Trial 5700...
Trial 5800...
Trial 5900...
Trial 6000...
Trial 6100...
Trial 6200...
Trial 6300...
Trial 6400...
Trial 6500...
Trial 6600...
Trial 6700...
Trial 6800...
Trial 6900...
Trial 7000...
Trial 7100..